# Étape 1 : Analyse Exploratoire et Préparation des Données (EDA)

**Projet IA — Classification Robuste en Environnement Déséquilibré**  
Dataset : Credit Card Fraud Detection (Kaggle) — 284 807 transactions, ratio 1:150

## Objectifs
- Analyse de la colinéarité (matrice de corrélation + VIF)
- Feature Engineering avancé
- Visualisation et traitement du déséquilibre de classes
- Comparaison des stratégies de rééchantillonnage (class_weight, SMOTE, ADASYN, NearMiss)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.join(os.getcwd(), 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import display, Image

from config import *
print(f'Project base: {BASE_DIR}')
print(f'Data file   : {DATA_FILE}')
print(f'Output dir  : {OUTPUT_DIR}')

## 1.1 Chargement et inspection initiale

In [ ]:
from eda import load_and_inspect
df_raw = load_and_inspect(DATA_FILE)

In [ ]:
# Distribution des classes
counts = df_raw[TARGET_COLUMN].value_counts()
print('Répartition des classes:')
print(f'  Normal (0) : {counts[0]:>10,}  ({counts[0]/len(df_raw)*100:.4f}%)')
print(f'  Fraude (1) : {counts[1]:>10,}  ({counts[1]/len(df_raw)*100:.4f}%)')
print(f'  Ratio      : {counts[0]/counts[1]:.1f}:1  → déséquilibre extrême')

# Statistiques descriptives
display(df_raw[['Amount', 'Time', 'V1', 'V2', 'V3', 'V4', 'V14', 'Class']].describe().round(3))

## 1.2 Feature Engineering avancé

Nous créons des variables supplémentaires :
- **Amount_log** : log(1+Amount) pour corriger l'asymétrie
- **Hour_sin / Hour_cos** : encodage cyclique de l'heure (évite la discontinuité 23h→0h)
- **Amount_zscore_local** : z-score local par tranche horaire (contexte du montant)
- **V_norm, V_mean, V_std** : statistiques des composantes PCA
- **V4_V11, V14_V17, V3_V10** : interactions multiplicatives des features les plus discriminantes

In [ ]:
from eda import feature_engineering
df = feature_engineering(df_raw)
print(f'\nFeatures après engineering : {df.shape[1]-1} (sans la cible)')
print('Nouvelles features :', [c for c in df.columns if c not in df_raw.columns])

## 1.3 Visualisation du déséquilibre de classes

In [ ]:
from eda import plot_class_imbalance
plot_class_imbalance(df, OUTPUT_DIR)
display(Image(filename=os.path.join(OUTPUT_DIR, 'eda_overview.png')))

## 1.4 Analyse de la colinéarité

### Matrice de corrélation
Identifie les paires de features fortement corrélées (|r| > 0.7).

### VIF (Variance Inflation Factor)
- VIF < 5 : pas de multicolinéarité
- 5 ≤ VIF < 10 : multicolinéarité modérée
- VIF ≥ 10 : multicolinéarité sévère → problème pour la régression logistique

In [ ]:
from eda import analyze_correlations
vif_df = analyze_correlations(df, OUTPUT_DIR)

print('\nTop 10 VIF :')
display(vif_df.head(10))

print('\nMatrice de corrélation (top 20 features):')
display(Image(filename=os.path.join(OUTPUT_DIR, 'correlation_matrix.png')))

print('\nAnalyse VIF :')
display(Image(filename=os.path.join(OUTPUT_DIR, 'vif_analysis.png')))

## 1.5 Préparation des données (Split + Scaling)

In [ ]:
from eda import prepare_data
(
    X_train, X_val, X_test,
    X_train_raw, X_val_raw, X_test_raw,
    y_train, y_val, y_test,
    feature_cols, scaler
) = prepare_data(df)

print(f'\nDimensions finales :')
print(f'  X_train : {X_train.shape}  | fraudes : {y_train.sum()} ({y_train.mean():.3%})')
print(f'  X_val   : {X_val.shape}  | fraudes : {y_val.sum()} ({y_val.mean():.3%})')
print(f'  X_test  : {X_test.shape}  | fraudes : {y_test.sum()} ({y_test.mean():.3%})')
print(f'  Scaler  : RobustScaler (robuste aux outliers, utilise la médiane/IQR)')

## 1.6 Comparaison des stratégies de rééchantillonnage

| Stratégie | Type | Avantages | Inconvénients |
|-----------|------|-----------|---------------|
| **class_weight** | Algorithmique | Pas de modification des données | Dépend du modèle |
| **SMOTE** | Sur-échantillonnage | Crée des exemples synthétiques | Peut créer du bruit |
| **ADASYN** | Sur-échantillonnage adaptatif | Focus sur les zones difficiles | Plus complexe |
| **NearMiss** | Sous-échantillonnage | Réduit le dataset | Perd beaucoup d'info |

In [ ]:
from eda import compare_resampling_strategies
resampling = compare_resampling_strategies(X_train, y_train, OUTPUT_DIR)
display(Image(filename=os.path.join(OUTPUT_DIR, 'resampling_comparison.png')))

## Résumé de l'étape 1

- **284 807 transactions** : 284 315 normales (99.83%) + 492 fraudes (0.17%)
- **Ratio d'imbalance** : 578:1 — déséquilibre extrême
- **Features** : 28 composantes PCA (V1-V28) + 14 features engineerées = **42 features**
- **Multicolinéarité** : VIF < 10 pour tous les features → pas de problème sévère
- **Split stratifié** : 70% train / 10% val / 20% test (la stratification préserve le ratio)
- **Scaler** : RobustScaler (médiane + IQR, robuste aux valeurs extrêmes des montants)